# IE Teachers Knowledge Graph Notebook
This Colab-first notebook walks through parsing IE teacher biographies, extracting entities, normalising labels, and building a NetworkX knowledge graph with friendly QA checkpoints.


In [ ]:
# Install dependencies (Restart & Run-All safe)
!pip install -q -r /content/ie-teachers-kg/requirements.txt


In [ ]:
# Global imports and deterministic setup
import os
import sys
import random
import json
from datetime import datetime
from collections import Counter

import numpy as np

random.seed(42)
np.random.seed(42)

REPO = "/content/ie-teachers-kg"
if REPO not in sys.path:
    sys.path.append(REPO)
SRC_PATH = os.path.join(REPO, "src")
if os.path.isdir(SRC_PATH) and SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

os.makedirs(os.path.join(REPO, "outputs"), exist_ok=True)
print("Repo path:", REPO)
print("Output dir:", os.path.join(REPO, "outputs"))


## Section A — Setup
We install the lightweight NLP stack (transformers + spaCy + NetworkX) and register the repo on `sys.path` so `src/` helpers are importable inside Colab.


## Section B — Load Data


In [ ]:
import pandas as pd

DATA_PATH = os.path.join(REPO, "data", "teachers_db_practice.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} professor bios from {DATA_PATH}")
df[['alias', 'area', 'position', 'full_info']].head()


## Section C — Sectioning, NER, and rule-based extraction
We clean HTML, segment the bios into logical sections, run two multilingual NER models, merge their outputs, and enrich the detections with regex rules (degrees, years, courses).


In [ ]:
from tqdm.auto import tqdm

from parsing import clean_html, split_sections, sentences
from ner_hf import load_pipelines, run_ner, merge_entities
from rules import extract_degrees_and_years, extract_courses, classify_org, year_bin

pipes = load_pipelines()
print("Loaded pipelines:", list(pipes.keys()))


In [ ]:
from typing import List, Dict, Tuple, Optional

SECTION_HINT = {
    'corporate_experience': 'company',
    'academic_experience': 'university',
    'academic_background': 'university',
}


def sentence_spans(text: str) -> List[Tuple[int, int, str]]:
    spans = []
    cursor = 0
    for sent in sentences(text):
        idx = text.find(sent, cursor)
        if idx == -1:
            idx = text.find(sent)
        if idx == -1:
            continue
        spans.append((idx, idx + len(sent), sent))
        cursor = idx + len(sent)
    return spans


def same_sentence(ent_a: Dict, ent_b: Dict, spans: List[Tuple[int, int, str]]) -> bool:
    a_center = (ent_a['start'] + ent_a['end']) / 2
    b_center = (ent_b['start'] + ent_b['end']) / 2
    for start, end, _ in spans:
        if start <= a_center <= end and start <= b_center <= end:
            return True
    return False


def nearest_org(location: Dict, orgs: List[Dict], spans: List[Tuple[int, int, str]]) -> Optional[Dict]:
    if not orgs:
        return None
    loc_center = (location['start'] + location['end']) / 2
    best = None
    best_dist = 1e9
    for org in orgs:
        if same_sentence(location, org, spans):
            org_center = (org['start'] + org['end']) / 2
            dist = abs(org_center - loc_center)
            if dist < best_dist:
                best = org
                best_dist = dist
    if best:
        return best
    for org in orgs:
        org_center = (org['start'] + org['end']) / 2
        dist = abs(org_center - loc_center)
        if dist < best_dist:
            best = org
            best_dist = dist
    return best


In [ ]:
extraction_records = []
university_names = []
company_names = []
location_names = []
all_courses = []

for idx, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
    html = getattr(row, 'full_info', '') or ''
    sections = split_sections(html)
    if not sections:
        sections = {'intro': clean_html(html)}
    prof_id = getattr(row, 'alias', f'prof_{idx}')
    record = {
        'prof_id': prof_id,
        'area': getattr(row, 'area', None),
        'position': getattr(row, 'position', None),
        'raw_sections': sections,
        'studies': [],
        'work': [],
        'courses': [],
        'locations': [],
    }

    for section_name, section_text in sections.items():
        text = section_text.strip()
        if not text:
            continue
        ents_a = run_ner(text, pipes['bert'])
        ents_b = run_ner(text, pipes['xlm'])
        merged = merge_entities(ents_a, ents_b)
        orgs = [e for e in merged if 'ORG' in (e['label'] or '').upper()]
        locs = [e for e in merged if (e['label'] or '').upper() in {'LOC', 'GPE'}]
        sent_sp = sentence_spans(text)

        org_loc_map = {}
        for loc in locs:
            target = nearest_org(loc, orgs, sent_sp)
            if target:
                org_loc_map.setdefault(target['text'], loc['text'])
                location_names.append(loc['text'])
                org_hint = SECTION_HINT.get(section_name) or classify_org(target['text'])
                record['locations'].append({
                    'org': target['text'],
                    'type': org_hint,
                    'location': loc['text'],
                    'source_section': section_name,
                })

        degree_info = extract_degrees_and_years(text)
        for item in degree_info:
            item['year_bin'] = year_bin(item.get('year'))
        degree_iter = iter(degree_info)

        courses = extract_courses(text)
        for course in courses:
            entry = {
                'course': course,
                'program': None,
                'source_section': section_name,
                'text_span': course,
            }
            record['courses'].append(entry)
            all_courses.append(course)

        for org in orgs:
            org_name = org['text']
            snippet = text[max(0, org['start'] - 50): org['end'] + 50]
            org_type = SECTION_HINT.get(section_name) or classify_org(org_name)
            if org_type == 'company':
                entry = {
                    'company': org_name,
                    'role': None,
                    'location': org_loc_map.get(org_name),
                    'source_section': section_name,
                    'text_span': snippet,
                }
                record['work'].append(entry)
                company_names.append(org_name)
            else:
                deg = next(degree_iter, None)
                entry = {
                    'university': org_name,
                    'degree': deg['degree'] if deg else None,
                    'field': deg['field'] if deg else None,
                    'year': deg['year'] if deg else None,
                    'year_bin': deg['year_bin'] if deg else None,
                    'location': org_loc_map.get(org_name),
                    'source_section': section_name,
                    'text_span': snippet,
                }
                record['studies'].append(entry)
                university_names.append(org_name)
    extraction_records.append(record)

print(f"Stored {len(extraction_records)} extraction records")


## Section D — Normalisation & Canonical labels
We clean organisation and location names, cluster near-duplicates with RapidFuzz, and map degrees to a controlled vocabulary.


In [ ]:
from normalize import normalize_name, cluster_and_canonicalize

CANON_DEGREES = {
    'PHD': 'PhD',
    'DOCTOR': 'PhD',
    'MSC': 'MSc',
    'MS': 'MSc',
    'MA': 'MA',
    'MBA': 'MBA',
    'BSC': 'BSc',
    'BA': 'BA',
    'MENG': 'MEng',
    'BENG': 'BEng',
}

uni_map = cluster_and_canonicalize(university_names) if university_names else {}
comp_map = cluster_and_canonicalize(company_names) if company_names else {}
loc_map = cluster_and_canonicalize(location_names) if location_names else {}

print("University clusters:", json.dumps(uni_map, indent=2)[:500])
print("Company clusters:", json.dumps(comp_map, indent=2)[:500])
print("Location clusters:", json.dumps(loc_map, indent=2)[:500])


def canonical_degree(name):
    if not name:
        return None
    key = name.upper().replace('.', '')
    return CANON_DEGREES.get(key)

for record in extraction_records:
    for study in record['studies']:
        uni_norm = normalize_name(study.get('university')) if study.get('university') else None
        loc_norm = normalize_name(study.get('location')) if study.get('location') else None
        study['university_norm'] = uni_norm
        study['university_canon'] = uni_map.get(uni_norm, uni_norm)
        study['location_norm'] = loc_norm
        study['location_canon'] = loc_map.get(loc_norm, loc_norm)
        study['degree_canon'] = canonical_degree(study.get('degree'))
    for work in record['work']:
        comp_norm = normalize_name(work.get('company')) if work.get('company') else None
        loc_norm = normalize_name(work.get('location')) if work.get('location') else None
        work['company_norm'] = comp_norm
        work['company_canon'] = comp_map.get(comp_norm, comp_norm)
        work['location_norm'] = loc_norm
        work['location_canon'] = loc_map.get(loc_norm, loc_norm)
    for loc in record['locations']:
        loc_norm = normalize_name(loc.get('location')) if loc.get('location') else None
        loc['location_norm'] = loc_norm
        loc['location_canon'] = loc_map.get(loc_norm, loc_norm)


## Section E — Graph building & artefacts


In [ ]:
from graph_utils import (
    new_graph,
    add_professor,
    add_university,
    add_company,
    add_course,
    add_degree,
    add_location,
    link_studied_at,
    link_worked_at,
    link_teaches,
    link_located_in,
    save_graph,
    top_k_by_degree,
)
import networkx as nx

G = new_graph()

for record in extraction_records:
    add_professor(G, record['prof_id'], area=record.get('area'), position=record.get('position'))
    for study in record['studies']:
        univ = study.get('university_canon') or study.get('university_norm')
        if not univ:
            continue
        add_university(G, univ)
        if study.get('degree_canon'):
            add_degree(G, study['degree_canon'], field=study.get('field'))
        if study.get('location_canon'):
            link_located_in(G, univ, study['location_canon'], 'university')
        link_studied_at(
            G,
            record['prof_id'],
            univ,
            degree=study.get('degree_canon') or study.get('degree'),
            field=study.get('field'),
            year=study.get('year'),
            year_bin=study.get('year_bin'),
            source_section=study.get('source_section', 'unknown'),
            text_span=study.get('text_span'),
        )
    for work in record['work']:
        comp = work.get('company_canon') or work.get('company_norm')
        if not comp:
            continue
        add_company(G, comp)
        if work.get('location_canon'):
            link_located_in(G, comp, work['location_canon'], 'company')
        link_worked_at(
            G,
            record['prof_id'],
            comp,
            role=work.get('role'),
            location=work.get('location_canon') or work.get('location'),
            source_section=work.get('source_section', 'unknown'),
            text_span=work.get('text_span'),
        )
    for course in record['courses']:
        name = course['course']
        if not name:
            continue
        add_course(G, name)
        link_teaches(
            G,
            record['prof_id'],
            name,
            program=course.get('program'),
            source_section=course.get('source_section', 'unknown'),
            text_span=course.get('text_span'),
        )

out_dir = os.path.join(REPO, 'outputs')
save_graph(G, out_dir)
print(f"Graph saved to {out_dir}")
print(nx.info(G))
print("Top universities:", top_k_by_degree(G, 'University'))
print("Top companies:", top_k_by_degree(G, 'Company'))


In [ ]:
import random
sampled = random.sample(extraction_records, min(10, len(extraction_records)))
for item in sampled:
    print(json.dumps(item, indent=2)[:1000])
    print('-' * 80)


## Section F — Quick visualisation


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

professors = [n for n, data in G.nodes(data=True) if data.get('type') == 'Professor']
selected = professors[:30]
sub_nodes = set(selected)
for prof in selected:
    sub_nodes.update(G.neighbors(prof))
H = G.subgraph(sub_nodes).copy()
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(H, seed=42)
color_map = []
for node in H:
    node_type = G.nodes[node].get('type')
    color_map.append({
        'Professor': '#1f77b4',
        'University': '#ff7f0e',
        'Company': '#2ca02c',
        'Course': '#d62728',
        'Location': '#9467bd',
        'Degree': '#8c564b',
    }.get(node_type, '#7f7f7f'))
nx.draw(H, pos, with_labels=False, node_color=color_map, node_size=120)
plt.title('Mini knowledge subgraph (first ~30 professors)')
plt.show()


## Section G — Export ZIP deliverable


In [ ]:
import shutil
import tempfile

stamp = datetime.utcnow().strftime('%Y%m%d')
zip_base = os.path.join(REPO, f'ie-teachers-kg_submit_{stamp}')
with tempfile.TemporaryDirectory() as tmp:
    targets = [
        ('notebooks', 'main.ipynb'),
        ('data', 'teachers_db_practice.csv'),
        ('outputs', 'nodes.csv'),
        ('outputs', 'edges.csv'),
        ('outputs', 'graph.gexf'),
        ('', 'requirements.txt'),
        ('', 'README.md'),
    ]
    for folder, fname in targets:
        src = os.path.join(REPO, folder, fname) if folder else os.path.join(REPO, fname)
        if os.path.exists(src):
            dst_dir = os.path.join(tmp, folder) if folder else tmp
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(src, os.path.join(dst_dir, fname))
    shutil.make_archive(zip_base, 'zip', tmp)

print(f"Created archive: {zip_base}.zip")


In [ ]:
try:
    from google.colab import files
    files.download(f"{zip_base}.zip")
except Exception as err:
    print("Download hint: run this cell in Colab to download the ZIP.")
    print(err)


## Section H — Documentation


**Pipeline pseudocode**
```
load CSV → iterate rows
  clean HTML → split sections
  run both NER models → merge spans
  attach regex degrees/courses + nearest locations
  normalise org/location strings via RapidFuzz clusters
  populate NetworkX graph with nodes + edges + provenance
persist nodes/edges/gexf → QA prints → build Colab ZIP deliverable
```


In [ ]:
degree_counter = Counter([
    study.get('degree_canon') or study.get('degree')
    for record in extraction_records for study in record['studies']
    if study.get('degree_canon') or study.get('degree')
])
course_counter = Counter(all_courses)
findings = [
    f"Processed {len(extraction_records)} professors with {len(G.nodes())} nodes and {len(G.edges())} edges in the KG.",
    f"Most common degrees: {degree_counter.most_common(3)}",
    f"Top courses mentioned: {course_counter.most_common(3)}",
    f"Top universities by degree centrality: {top_k_by_degree(G, 'University')[:3]}",
]
for item in findings:
    print(f"- {item}")
